Builds the skills_dictionary reference table (Day 11).
Writes: jobmarket.silver.skills_dictionary
Schema: skill, skill_group, aliases ARRAY, needs_boundary BOOLEAN
Seeding: hand-curated core + O*NET supplement + soft skills.
Iterated on Day 13 after first extraction results.

In [0]:
from pyspark.sql import functions as F

In [0]:
# (skill, group, aliases, needs_boundary)
# needs_boundary=True -> Day 12 must use \b regex, not contains()
core_skills = [
    # --- programming ---
    ("python",     "programming", ["python"], False),
    ("sql",        "programming", ["sql"], True),          # 3 chars, but also: "mysql" contains it — boundary handles
    ("r", "programming",
     ["r programming", "r language", "in r", "r studio", "rstudio",
      "python, r", "r, python", "python/r", "r/python", "sql, r", "r, sql"],
     True),            # the classic trap
    ("java",       "programming", ["java"], True),         # boundary keeps "javascript" out
    ("javascript", "programming", ["javascript", "js"], True),   # "js" is short
    ("typescript", "programming", ["typescript"], False),
    ("scala",      "programming", ["scala"], True),        # "scalable" contains it!
    ("c++",        "programming", ["c++"], True),
    ("c#",         "programming", ["c#", ".net", "dotnet"], True),
    ("go",         "programming", ["golang"], False),      # NOTE: alias is golang ONLY —
                                                           # bare "go" is hopeless even with boundaries
                                                           # ("go to", "go-getter"); we accept undercounting
    ("rust",       "programming", ["rust"], True),         # "trust"? no — boundary suffices; but rust-the-corrosion exists in industrial postings
    ("bash",       "programming", ["bash", "shell scripting"], True),

    # --- data platforms & engineering ---
    ("spark",      "data_platform", ["spark", "pyspark", "apache spark"], True),
    ("databricks", "data_platform", ["databricks"], False),
    ("snowflake",  "data_platform", ["snowflake"], False),
    ("dbt",        "data_platform", ["dbt"], True),
    ("airflow",    "data_platform", ["airflow", "apache airflow"], False),
    ("kafka",      "data_platform", ["kafka", "apache kafka"], False),
    ("hadoop",     "data_platform", ["hadoop"], False),
    ("delta lake", "data_platform", ["delta lake"], False),
    ("etl",        "data_platform", ["etl", "elt"], True),
    ("data warehouse", "data_platform", ["data warehouse", "data warehousing", "dwh"], True),
    ("data modeling",  "data_platform", ["data modeling", "data modelling", "dimensional modeling"], False),
    ("postgresql", "data_platform", ["postgresql", "postgres"], False),
    ("mysql",      "data_platform", ["mysql"], False),
    ("mongodb",    "data_platform", ["mongodb", "mongo"], False),
    ("redis",      "data_platform", ["redis"], False),
    ("elasticsearch", "data_platform", ["elasticsearch", "elastic search","elk","kibana"], True),

    # --- BI & analytics ---
    ("tableau",    "bi_tool", ["tableau"], False),
    ("power bi",   "bi_tool", ["power bi", "powerbi", "power-bi"], False),
    ("looker",     "bi_tool", ["looker", "lookml"], False),
    ("excel",      "bi_tool", ["excel", "microsoft excel", "pivot tables"], True),   # "excel" the verb — boundary won't save us; Day 13 measures
    ("qlik",       "bi_tool", ["qlik", "qlikview", "qlik sense"], True),
    ("ssrs",       "bi_tool", ["ssrs"], True),
    ("a/b testing","bi_tool", ["a/b testing", "ab testing", "experimentation"], False),

    # --- cloud ---
    ("aws",        "cloud", ["aws", "amazon web services"], True),
    ("azure",      "cloud", ["azure", "microsoft azure"], False),
    ("gcp",        "cloud", ["gcp", "google cloud"], True),
    ("s3",         "cloud", ["s3"], True),
    ("lambda",     "cloud", ["aws lambda"], False),        # bare "lambda" collides with Python lambdas
    ("redshift",   "cloud", ["redshift"], False),
    ("bigquery",   "cloud", ["bigquery", "big query"], False),
    ("docker",     "cloud", ["docker", "containerization"], False),
    ("kubernetes", "cloud", ["kubernetes", "k8s"], True),
    ("terraform",  "cloud", ["terraform"], False),
    ("ci/cd",      "cloud", ["ci/cd", "cicd", "continuous integration"], False),
    ("git",        "cloud", ["git", "github", "gitlab"], True),

    # --- ML & AI ---
    ("machine learning", "ml_ai", ["machine learning", "ml"], True),
    ("deep learning",    "ml_ai", ["deep learning", "neural network", "neural networks"], False),
    ("nlp",        "ml_ai", ["nlp", "natural language processing"], True),
    ("computer vision", "ml_ai", ["computer vision"], False),
    ("tensorflow", "ml_ai", ["tensorflow"], False),
    ("pytorch",    "ml_ai", ["pytorch"], False),
    ("scikit-learn", "ml_ai", ["scikit-learn", "sklearn", "scikit learn"], False),
    ("pandas",     "ml_ai", ["pandas"], False),
    ("numpy",      "ml_ai", ["numpy"], False),
    ("llm",        "ml_ai", ["llm", "llms", "large language model", "large language models", "generative ai", "genai"], True),
    ("mlops",      "ml_ai", ["mlops", "ml ops"], False),
    ("statistics", "ml_ai", ["statistics", "statistical analysis", "statistical modeling"], False),

    # --- soft skills ---
    ("communication", "soft", ["communication skills", "written and verbal"], False),
    ("leadership",    "soft", ["leadership"], False),
    ("stakeholder management", "soft", ["stakeholder management", "stakeholders"], False),
    ("project management", "soft", ["project management"], False),
    ("agile",      "soft", ["agile", "scrum", "kanban"], True),
    ("problem solving", "soft", ["problem solving", "problem-solving"], False),
    ("mentoring",  "soft", ["mentoring", "mentorship"], False),

    # --- frontend & web (LLM-comparison finds, Day 13) ---
    ("html",        "programming", ["html", "html5"], True),
    ("css",         "programming", ["css", "css3"], True),
    ("angular",     "programming", ["angular", "angularjs"], False),
    ("vue",         "programming", ["vue", "vue.js", "vuejs"], True),
    ("spring boot", "programming", ["spring boot", "spring framework"], False),

    # web/backend frameworks (they appear constantly in SWE postings)
    ("react",      "programming", ["react", "react.js", "reactjs"], False),
    ("node.js",    "programming", ["node.js", "nodejs", "node js"], False),
    ("django",     "programming", ["django"], False),
    ("fastapi",    "programming", ["fastapi"], False),
    ("rest api",   "programming", ["rest api", "restful", "rest apis", "api development"], False),
    ("graphql",    "programming", ["graphql"], False),

    # data tools we somehow skipped
    ("streamlit",  "bi_tool", ["streamlit"], False),
    ("ssis",       "data_platform", ["ssis"], True),
    ("sas",        "programming", ["sas"], True),          # boundary: "sassy", names
    ("matlab",     "programming", ["matlab"], False),
    ("oracle",     "data_platform", ["oracle", "pl/sql", "plsql"], False),
    ("sql server", "data_platform", ["sql server", "t-sql", "tsql", "microsoft sql"], False),

    # enterprise & workflow (huge in the all-industries Kaggle data)
    ("salesforce", "data_platform", ["salesforce"], False),
    ("sap",        "data_platform", ["sap"], True),
    ("jira",       "soft", ["jira"], False),
    ("confluence", "soft", ["confluence"], False),

    # analytics adjacents
    ("data visualization", "bi_tool", ["data visualization", "data viz", "dashboarding"], False),
    ("data quality", "data_platform", ["data quality", "data validation"], False),
    ("data governance", "data_platform", ["data governance"], False),
]

print(f"Core skills: {len(core_skills)}")

# --- O*NET-equivalent supplement: enterprise analytics, data tooling,
# --- and infrastructure-adjacent skills common in postings
supplement = [

    ("microservices", "programming", ["microservices", "microservice"], False),
    ("testing",       "programming", ["unit testing", "test automation",
                       "automated testing", "integration testing", "tdd",
                       "test-driven"], False),
    
    # analytics & statistics tools
    ("alteryx",       "bi_tool", ["alteryx"], False),
    ("spss",          "bi_tool", ["spss"], True),
    ("stata",         "bi_tool", ["stata"], True),
    ("microstrategy", "bi_tool", ["microstrategy"], False),
    ("cognos",        "bi_tool", ["cognos", "ibm cognos"], False),
    ("domo",          "bi_tool", ["domo"], True),
    ("sisense",       "bi_tool", ["sisense"], False),
    ("google analytics", "bi_tool", ["google analytics", "ga4"], True),

    # data integration & pipelines
    ("informatica",   "data_platform", ["informatica"], False),
    ("talend",        "data_platform", ["talend"], False),
    ("fivetran",      "data_platform", ["fivetran"], False),
    ("airbyte",       "data_platform", ["airbyte"], False),
    ("mulesoft",      "data_platform", ["mulesoft"], False),
    ("apache nifi",   "data_platform", ["nifi", "apache nifi"], True),
    ("flink",         "data_platform", ["flink", "apache flink"], True),
    ("presto",        "data_platform", ["presto", "trino"], True),
    ("hive",          "data_platform", ["hive", "apache hive"], True),   # "beehive", "hive mind"
    ("teradata",      "data_platform", ["teradata"], False),
    ("cassandra",     "data_platform", ["cassandra"], True),             # it's also a name
    ("dynamodb",      "data_platform", ["dynamodb", "dynamo db"], False),

    # enterprise platforms
    ("workday", "data_platform",
     ["workday hcm", "workday integration", "workday experience",
      "experience with workday", "workday reporting",
      "salesforce and workday", "workday and salesforce"],
     False),               # "workday" the common noun!
    ("servicenow",    "data_platform", ["servicenow", "service now"], False),
    ("netsuite",      "data_platform", ["netsuite"], False),
    ("splunk",        "data_platform", ["splunk"], False),
    ("snowplow",      "data_platform", ["snowplow"], False),
    ("segment",       "data_platform", ["segment.io"], False),           # bare "segment" hopeless
    ("hubspot",       "data_platform", ["hubspot"], False),
    ("marketo",       "data_platform", ["marketo"], False),
    ("nosql",         "data_platform", ["nosql", "no-sql"], False),


    # infra & monitoring adjacents
    ("linux",         "cloud", ["linux", "unix"], False),
    ("jenkins",       "cloud", ["jenkins"], True),                       # also a surname
    ("grafana",       "cloud", ["grafana"], False),
    ("prometheus",    "cloud", ["prometheus"], True),                    # Greek titan, movie
    ("datadog",       "cloud", ["datadog"], False),
    ("ansible",       "cloud", ["ansible"], False),
    ("devops",        "cloud", ["devops", "devsecops", "dev ops"], False),
    

    # ML/AI extras
    ("hugging face",  "ml_ai", ["hugging face", "huggingface"], False),
    ("langchain",     "ml_ai", ["langchain"], False),
    ("xgboost",       "ml_ai", ["xgboost"], False),
    ("keras",         "ml_ai", ["keras"], False),
    ("openai",        "ml_ai", ["openai", "gpt-4", "gpt"], True),
    ("rag",           "ml_ai", ["retrieval augmented generation", "retrieval-augmented"], False),  # bare "rag" hopeless
]

all_skills = core_skills + supplement
print(f"Total skills: {len(all_skills)}")

In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                ArrayType, BooleanType)

schema = StructType([
    StructField("skill",          StringType(),  False),
    StructField("skill_group",    StringType(),  False),
    StructField("aliases",        ArrayType(StringType()), False),
    StructField("needs_boundary", BooleanType(), False),
])

dict_df = spark.createDataFrame(all_skills, schema)

(dict_df.write.format("delta").mode("overwrite")
    .saveAsTable("jobmarket.silver.skills_dictionary"))

d = spark.table("jobmarket.silver.skills_dictionary")

# verification suite
print("Rows:", d.count())
d.groupBy("skill_group").count().orderBy(F.desc("count")).display()

# duplicate canonical names?
dupes = d.groupBy("skill").count().filter("count > 1")
print("Duplicate skills:", dupes.count())

# any alias under two different skills? (the double-count bug class)
alias_dupes = (d.select("skill", F.explode("aliases").alias("alias"))
    .groupBy("alias").agg(F.countDistinct("skill").alias("n"))
    .filter("n > 1"))
print("Aliases under multiple skills:", alias_dupes.count())
alias_dupes.display()

# short aliases not flagged for boundaries?
unflagged = (d.select("skill", "needs_boundary", F.explode("aliases").alias("alias"))
    .filter((F.length("alias") <= 3) & (~F.col("needs_boundary"))))
print("Short aliases missing boundary flag:", unflagged.count())
unflagged.display()